In [1]:
import os
import time
import json
import requests
import pandas as pd
import sqlite3
from datetime import datetime
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm


In [13]:
BASE_DIR = "/home/shail/stockshortlisting"
BASE_DIR_db = "/home/shail/db"


"""
Fetch equity + derivative data for a given NSE stock symbol.
Returns a list of values or Nones if request fails.
"""

now = datetime.now()
current_year = now.year
#current_month = now.month
current_month = 3
current_time = now.strftime('%H:%M:%S')
current_time = now.strftime('%H:%M')
current_date = now.strftime('%Y-%m-%d')
stock = 'TCS'
encoded_stock = stock.replace('&', '%26')
main_url = 'https://www.nseindia.com'
qty_url = f"https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi?functionName=getSymbolData&marketType=N&series=EQ&symbol={encoded_stock}"
fno_url = f'https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi?functionName=getSymbolDerivativesData&symbol={encoded_stock}'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/91.0.4472.124 Safari/537.36',
    'Referer': f'https://www.nseindia.com/get-quotes/equity?symbol={encoded_stock}',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5'
}

session = requests.Session()
retry = Retry(
    total=2,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

try:
    session.get(main_url, headers=headers)
    time.sleep(1)

    response = session.get(qty_url, headers=headers)
    response2 = session.get(fno_url, headers=headers)
    response.raise_for_status()
    eqjson = response.json()
    fno = response2.json()

    all_futstk_data = [item for item in fno.get('data', []) if item.get('instrumentType') == 'FUTSTK']

    for item in all_futstk_data:
        expiry_date_str = item.get('expiryDate')
        expiry_date = datetime.strptime(expiry_date_str, '%d-%b-%Y')

        if expiry_date.year == current_year and expiry_date.month == current_month:
            current_month_identifier = item['identifier']
            fustk_encoded = current_month_identifier.replace('&', '%26')

            fubuysell_url = (
                f'https://www.nseindia.com/api/NextApi/apiClient/GetQuoteApi?'
                f'functionName=getTradeInfoDerivative&symbol={encoded_stock}&identifier={fustk_encoded}'
            )

            fuorder_resp = session.get(fubuysell_url, headers=headers)
            fuorder_resp.raise_for_status()
            fustkjson = fuorder_resp.json()
            dt_str = fustkjson['derivateResponse'][0]['lastUpdateTime'] 
            date_part, fetch_time = dt_str.split()
            listnew = [
                stock, current_date, current_time,fetch_time,
                eqjson.get('equityResponse', [{}])[0].get('orderBook', {}).get('totalBuyQuantity', None),
                eqjson.get('equityResponse', [{}])[0].get('orderBook', {}).get('totalSellQuantity', None),
                eqjson.get('equityResponse', [{}])[0].get('tradeInfo', {}).get('totalTradedValue', None),
                eqjson.get('equityResponse', [{}])[0].get('tradeInfo', {}).get('totalTradedVolume', None),
                eqjson.get('equityResponse', [{}])[0].get('metaData', {}).get('dayHigh', None),
                eqjson.get('equityResponse', [{}])[0].get('metaData', {}).get('change', None),
                eqjson.get('equityResponse', [{}])[0].get('metaData', {}).get('pChange', None),
                eqjson.get('equityResponse', [{}])[0].get('metaData', {}).get('previousClose', None),
                eqjson.get('equityResponse', [{}])[0].get('metaData', {}).get('averagePrice', None),
                fustkjson.get('derivateResponse', [{}])[0].get('metaData', {}).get('last', None),
                fustkjson.get('derivateResponse', [{}])[0].get('metaData', {}).get('change', None),
                fustkjson.get('derivateResponse', [{}])[0].get('metaData', {}).get('perchange', None),
                fustkjson.get('derivateResponse', [{}])[0].get('orderBook', {}).get('totalSellQuantity', None),
                fustkjson.get('derivateResponse', [{}])[0].get('orderBook', {}).get('totalBuyQuantity', None),
                fustkjson.get('derivateResponse', [{}])[0].get('tradeInfo', {}).get('totalTradedVolume', None),
                fustkjson.get('derivateResponse', [{}])[0].get('tradeInfo', {}).get('totalTradedValue', None),
                fustkjson.get('derivateResponse', [{}])[0].get('tradeInfo', {}).get('openinterest', None)
            ]
            #return listnew

    # If no matching expiry found
    #return [stock] + [None] * 19

except requests.exceptions.RequestException as e:
    #print(f"An error occurred for {stock}: {e}")
    print(f"An error occurred for : {e}")
    #return [stock] + [None] * 19
print("Completed the Task")


Completed the Task


In [4]:
fustkjson

{'derivateResponse': [{'orderBook': {'buyPrice1': '2629.3',
    'buyQuantity1': '175',
    'buyPrice2': '2629.2',
    'buyQuantity2': '175',
    'buyPrice3': '2629',
    'buyQuantity3': '175',
    'buyPrice4': '2628.7',
    'buyQuantity4': '350',
    'buyPrice5': '2628.6',
    'buyQuantity5': '175',
    'sellPrice1': '2629.8',
    'sellQuantity1': '175',
    'sellPrice2': '2629.9',
    'sellQuantity2': '175',
    'sellPrice3': '2630',
    'sellQuantity3': '350',
    'sellPrice4': '2630.4',
    'sellQuantity4': '175',
    'sellPrice5': '2630.5',
    'sellQuantity5': '175',
    'totalBuyQuantity': '123550',
    'totalSellQuantity': '237825',
    'perBuyQty': 34.19,
    'perSellQty': 65.81,
    'ccBestBuy': 5.710032826509693,
    'ccBestSell': 5.924131976150823,
    'ccLTP': 5.795677371655202},
   'metaData': {'identifier': 'FUTSTKTCS30-03-2026XX0.00',
    'symbol': 'TCS',
    'last': '2629.5',
    'change': '41.6',
    'perchange': '1.60748096912554580934348313304223501681E00',
    'open

In [8]:
for i in fustkjson['derivateResponse']:
    print (i)

{'orderBook': {'buyPrice1': '2629.3', 'buyQuantity1': '175', 'buyPrice2': '2629.2', 'buyQuantity2': '175', 'buyPrice3': '2629', 'buyQuantity3': '175', 'buyPrice4': '2628.7', 'buyQuantity4': '350', 'buyPrice5': '2628.6', 'buyQuantity5': '175', 'sellPrice1': '2629.8', 'sellQuantity1': '175', 'sellPrice2': '2629.9', 'sellQuantity2': '175', 'sellPrice3': '2630', 'sellQuantity3': '350', 'sellPrice4': '2630.4', 'sellQuantity4': '175', 'sellPrice5': '2630.5', 'sellQuantity5': '175', 'totalBuyQuantity': '123550', 'totalSellQuantity': '237825', 'perBuyQty': 34.19, 'perSellQty': 65.81, 'ccBestBuy': 5.710032826509693, 'ccBestSell': 5.924131976150823, 'ccLTP': 5.795677371655202}, 'metaData': {'identifier': 'FUTSTKTCS30-03-2026XX0.00', 'symbol': 'TCS', 'last': '2629.5', 'change': '41.6', 'perchange': '1.60748096912554580934348313304223501681E00', 'open': '2620', 'high': '2664.8', 'low': '2605.1', 'strikeprice': '       0.00', 'optiontype': 'XX', 'expiry': '30-03-2026', 'instrumenttype': 'FUTSTK', '

In [11]:
fustkjson['derivateResponse'][0]['lastUpdateTime']

'25-Feb-2026 13:59'

In [12]:
dt_str = fustkjson['derivateResponse'][0]['lastUpdateTime'] 
date_part, fetch_time = dt_str.split()
fetch_time

'13:59'